# آموزش، ارزیابی و تجربه‌ی کاربردی روی GANwriting فارسی (دیتاست آبان)
### قابل اجرا روی Google Colab یا Kaggle (GPU رایگان)

این نوت‌بوک، پایپ‌لاین GAN موجود در این ریپو (`network_tro.py`, `modules_tro.py`, `main_run.py`, `load_data.py`, ...) را که از قبل برای دیتاست **آبان** (۵۰۰ نویسنده × ۱۲۵ کلمه، هر کلمه یک‌بار) تطبیق داده شده، در محیط کولب/کگل اجرا می‌کند و یک تجربه‌ی عملی روی آن می‌سازد.

## موضوع پیشنهادی
> **«تولید دست‌خط فارسی سبک‌محور (style-conditioned) روی دیتاست آبان، و بررسی این‌که آیا داده‌ی مصنوعیِ تولیدشده با آن، دقت بازشناسی دست‌خط فارسی را در رژیم کم‌داده بهبود می‌دهد؟»**

این چارچوب، هم از زیرساخت مدل تولیدی که از قبل در ریپو پیاده‌سازی و دیباگ شده استفاده می‌کند (کم‌ریسک‌تر از نوشتن معماری جدید از صفر)، و هم یک یافته‌ی کمّی و قابل‌دفاع تولید می‌کند (بهبود/عدم‌بهبود CER یک بازشناس با و بدون داده‌ی مصنوعی).

## پیش‌نیازها
- یک GPU رایگان: در **Kaggle** از `Settings → Accelerator → GPU T4 x2`، در **Colab** از `Runtime → Change runtime type → GPU` استفاده کنید.
- تصاویر PNG کلمات دیتاست آبان (بخش ۲ توضیح می‌دهد چطور تهیه‌شان کنید).

## ساختار نوت‌بوک
1. راه‌اندازی محیط و کلون ریپو
2. آماده‌سازی دیتاست
3. تست سلامت سریع (Smoke Test) — قبل از هر آموزش طولانی اجرا کنید
4. آموزش با بودجه‌ی زمانی (بین سشن‌های کولب/کگل قابل ازسرگیری)
5. ارزیابی: CER بازشناسی، FID، سازگاری سبک نویسنده
6. دمو: تولید کلمه با سبک یک نویسنده
7. تجربه‌ی کاربردی: آیا داده‌ی مصنوعی، یک بازشناس مستقل را بهتر می‌کند؟
8. جمع‌بندی، عیب‌یابی و نکات نگارش پایان‌نامه

⚠️ **توجه:** این نوت‌بوک در محیطی بدون GPU و بدون خود دیتاست نوشته شده (امکان اجرای واقعی برای تست در دسترس نبود). منطق کد بر پایه‌ی بررسی دقیق فایل‌های موجود در ریپو (`main_run.py`, `network_tro.py`, `modules_tro.py`, `load_data.py`) نوشته شده، اما ممکن است هنگام اولین اجرا نیاز به اصلاحات جزئی (نسخه‌ی کتابخانه‌ها و…) داشته باشید. حتماً از **بخش ۳ (Smoke Test)** قبل از commit کردن به یک آموزش طولانی استفاده کنید.

In [ ]:
import torch, sys, os, platform

print("Python:", platform.python_version())
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU پیدا نشد! در Kaggle: Settings > Accelerator > GPU T4 x2 را فعال کنید. "
        "در Colab: Runtime > Change runtime type > GPU را انتخاب کنید. سپس این سلول را دوباره اجرا کنید."
    )
print("GPU:", torch.cuda.get_device_name(0))

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print("Kaggle:", IS_KAGGLE, "| Colab:", IS_COLAB)

In [ ]:
# نصب کتابخانه‌های اضافی مورد نیاز ریپو (بدون دست‌کاری torch/torchvision از پیش‌نصب‌شده)
%pip install -q albumentations python-Levenshtein arabic-reshaper python-bidi gdown torchmetrics torch-fidelity opencv-python-headless

In [ ]:
import subprocess

REPO_URL = "https://github.com/mo0o0o0os/persian-ganwriting.git"
REPO_DIR = "/kaggle/working/persian-ganwriting" if IS_KAGGLE else "/content/persian-ganwriting"

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # پوشه‌ای در گوگل‌درایو برای نگهداری چک‌پوینت‌ها بین سشن‌های کولب (که هر بار ریست می‌شوند)
    PERSIST_DIR = "/content/drive/MyDrive/persian_ganwriting_ckpt"
else:
    # /kaggle/working معمولاً بین ادیت‌های همین نوت‌بوک باقی می‌ماند، اما تضمین‌شده نیست؛
    # در پایان هر سشن حتماً save_weights/logs را Export یا Commit کنید (بخش ۸).
    PERSIST_DIR = "/kaggle/working/persist_ckpt"

os.makedirs(PERSIST_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

# save_weights / logs / imgs را به مسیر پایدار لینک می‌کنیم تا بین اجراها گم نشوند
for sub in ["save_weights", "logs", "imgs"]:
    target = os.path.join(PERSIST_DIR, sub)
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(sub) and not os.path.exists(sub):
        os.symlink(target, sub)
print("save_weights/logs/imgs -> ", PERSIST_DIR)

## بخش ۲ — آماده‌سازی دیتاست

تصاویر باید در مسیر `datasets/aban/words/<idx>.png` قرار بگیرند (همان مسیری که `load_data.py` انتظار دارد). سه روش را در `DATA_CONFIG` زیر می‌بینید — **دقیقاً یکی** را با توجه به این‌که دیتاست را کجا نگه می‌دارید پر کنید:

- **روش ۱ (خودکار):** دانلود از گوگل‌درایو با همان File ID که در `download_dataset_farsi.sh` ریپو آمده. ⚠️ این ID تأییدنشده است — ممکن است قدیمی/نامعتبر باشد یا دسترسی عمومی نداشته باشد. اگر کار نکرد، روش ۲ یا ۳ را استفاده کنید.
- **روش ۲ (Kaggle):** پوشه‌ی `words` را زیپ کرده و به‌عنوان یک Kaggle Dataset آپلود کنید، آن را Add Data کنید، و مسیرش را (مثل `/kaggle/input/aban-words/words`) در `kaggle_dataset_path` بگذارید.
- **روش ۳ (Colab):** اگر تصاویر را در گوگل‌درایوِ خودتان دارید، مسیر پوشه‌ی `words` را در `drive_words_path` بگذارید.

سلول بعد تعداد تصاویر یافت‌شده را با تعداد مورد انتظار (طبق فایل‌های Groundtruth) مقایسه می‌کند تا از سلامت دیتاست مطمئن شوید.

In [ ]:
DATA_CONFIG = {
    "gdrive_file_id": "1wKrSQHif96ucColaRkebKTQUZ84g1weY",  # از download_dataset_farsi.sh - قبل از اعتماد، تأیید کنید
    "kaggle_dataset_path": None,   # مثال: "/kaggle/input/aban-words/words"
    "drive_words_path": None,      # مثال: "/content/drive/MyDrive/aban_dataset/words"
}

TARGET_DIR = "datasets/aban/words"
os.makedirs("datasets/aban", exist_ok=True)

def count_expected_images():
    n = 0
    for f in ["Groundtruth_farsi/gan.aban.tr_va.gt.filter27", "Groundtruth_farsi/gan.aban.test.gt.filter27"]:
        with open(f, encoding="utf-8") as fh:
            n += len(fh.readlines())
    return n

expected = count_expected_images()
print(f"تعداد نمونه‌ی مورد انتظار طبق فایل‌های Groundtruth: {expected}")

if DATA_CONFIG["kaggle_dataset_path"] and not os.path.exists(TARGET_DIR):
    os.symlink(DATA_CONFIG["kaggle_dataset_path"], TARGET_DIR)
elif DATA_CONFIG["drive_words_path"] and not os.path.exists(TARGET_DIR):
    os.symlink(DATA_CONFIG["drive_words_path"], TARGET_DIR)
elif DATA_CONFIG["gdrive_file_id"] and not os.path.exists(TARGET_DIR):
    print("در حال تلاش برای دانلود خودکار از گوگل‌درایو ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "unrar", "p7zip-full"], check=False)
    rar_path = "datasets/aban/TC_Words.rar"
    r = subprocess.run(["gdown", "--output", rar_path,
                         f"https://drive.google.com/uc?id={DATA_CONFIG['gdrive_file_id']}"])
    if r.returncode == 0 and os.path.exists(rar_path):
        subprocess.run(["unrar", "x", "-y", rar_path, "datasets/aban/"], check=False)
        for name in os.listdir("datasets/aban"):
            full = os.path.join("datasets/aban", name)
            if os.path.isdir(full) and name != "words":
                pngs = [x for x in os.listdir(full) if x.lower().endswith(".png")]
                if pngs:
                    os.rename(full, TARGET_DIR)
                    break
    else:
        print("⚠️ دانلود خودکار ناموفق بود؛ باید دیتاست را با روش ۲ یا ۳ دستی فراهم کنید.")

if os.path.isdir(TARGET_DIR):
    actual = len([f for f in os.listdir(TARGET_DIR) if f.lower().endswith(".png")])
    print(f"تعداد تصاویر یافت‌شده در {TARGET_DIR}: {actual} / انتظار: {expected}")
    if actual < 0.9 * expected:
        print("⚠️ تعداد تصاویر به‌طور محسوسی کمتر از انتظار است — دیتاست را بررسی کنید.")
else:
    raise FileNotFoundError(
        f"پوشه‌ی {TARGET_DIR} پیدا نشد. یکی از سه روش در DATA_CONFIG را تنظیم کنید و دوباره اجرا کنید."
    )

In [ ]:
import cv2, random, matplotlib.pyplot as plt
from arabic_reshaper import reshape as ar_reshape
from bidi.algorithm import get_display

with open("Groundtruth_farsi/gan.aban.tr_va.gt.filter27", encoding="utf-8") as f:
    _lines = [l.strip().split(" ", 1) for l in f if l.strip()]

sample = random.sample(_lines, 6)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, (idpart, label) in zip(axes, sample):
    wid, idx = idpart.split(",")
    img = cv2.imread(os.path.join(TARGET_DIR, idx + ".png"), 0)
    ax.imshow(img, cmap="gray")
    ax.set_title(get_display(ar_reshape(label)), fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

## بخش ۳ — تست سلامت (Smoke Test)

پیش از هر آموزش طولانی، یک بچ کوچک را از تمام مسیرهای مدل (recognizer، classifier، discriminator، generator) عبور می‌دهیم تا مشکلات نسخه‌ی کتابخانه یا مسیر دیتاست زود مشخص شوند، نه بعد از ساعت‌ها GPU رایگان.

In [ ]:
import numpy as np

def sort_batch(batch):
    # دقیقاً معادل sort_batch در main_run.py — این‌جا هم لازم داریم چون main_run.py را
    # مستقیم import نمی‌کنیم (به‌خاطر argparse در سطح ماژول آن)
    train_domain, train_wid, train_idx, train_img = [], [], [], []
    train_img_width, train_label, img_xts, label_xts, label_xts_swap = [], [], [], [], []
    for (domain, wid, idx, img, img_width, label, img_xt, label_xt, label_xt_swap) in batch:
        train_domain.append(domain); train_wid.append(wid); train_idx.append(idx)
        train_img.append(img); train_img_width.append(img_width); train_label.append(label)
        img_xts.append(img_xt); label_xts.append(label_xt); label_xts_swap.append(label_xt_swap)
    train_wid = torch.from_numpy(np.array(train_wid, dtype="int64"))
    train_img = torch.from_numpy(np.array(train_img, dtype="float32"))
    train_img_width = torch.from_numpy(np.array(train_img_width, dtype="int64"))
    train_label = torch.from_numpy(np.array(train_label, dtype="int64"))
    img_xts = torch.from_numpy(np.array(img_xts, dtype="float32"))
    label_xts = torch.from_numpy(np.array(label_xts, dtype="int64"))
    label_xts_swap = torch.from_numpy(np.array(label_xts_swap, dtype="int64"))
    return (np.array(train_domain), train_wid, np.array(train_idx), train_img,
            train_img_width, train_label, img_xts, label_xts, label_xts_swap)

gpu = torch.device("cuda")

In [ ]:
from load_data import loadData, NUM_WRITERS
from network_tro import ConTranModel
from loss_tro import CER
from torch import optim

print("در حال بارگذاری دیتاست برای تست سلامت ...")
data_train, data_test = loadData(oov=True)
print("تعداد نویسندگان train/test:", len(data_train), len(data_test))

loader = torch.utils.data.DataLoader(data_train, batch_size=4, shuffle=True,
                                      collate_fn=sort_batch, num_workers=0)

model = ConTranModel(NUM_WRITERS, show_iter_num=500, oov=True).to(gpu)
print("تعداد پارامترهای مدل:", sum(p.numel() for p in model.parameters()))

dis_opt = optim.Adam(model.dis.parameters(), lr=8e-5)
gen_opt = optim.Adam(model.gen.parameters(), lr=8e-5)
rec_opt = optim.Adam(model.rec.parameters(), lr=8e-6)
cla_opt = optim.Adam(model.cla.parameters(), lr=8e-6)

batch = next(iter(loader))
rec_opt.zero_grad(); l_rec = model(batch, 0, "rec_update", CER()); rec_opt.step()
cla_opt.zero_grad(); l_cla = model(batch, 0, "cla_update");        cla_opt.step()
dis_opt.zero_grad(); l_dis = model(batch, 0, "dis_update");        dis_opt.step()
gen_opt.zero_grad(); l_tot, *_ = model(batch, 0, "gen_update", [CER(), CER()]); gen_opt.step()

print("✅ یک گام کامل (rec+cla+dis+gen) بدون خطا اجرا شد.")
print(f"l_rec={l_rec.item():.3f} | l_cla={l_cla.item():.3f} | l_dis={l_dis.item():.3f} | l_total={l_tot.item():.3f}")
print("اگر همه‌چیز عدد محدود (نه NaN/inf) چاپ شد، محیط آماده‌ی آموزش کامل است.")

## بخش ۴ — آموزش با بودجه‌ی زمانی (قابل ازسرگیری بین سشن‌ها)

آموزش کامل (`main_run.py`) به‌صورت subprocess اجرا می‌شود — دقیقاً همان کد اصلی ریپو، بدون تغییر منطق. دو ثابتِ فاصله‌ی ذخیره/ارزیابی را کمی کاهش می‌دهیم تا با طول محدود سشن‌های رایگان سازگارتر شود (چک‌پوینت‌های نزدیک‌تر به هم یعنی از دست دادن کمتر در صورت قطع‌شدن سشن).

هر بار این سلول‌ها را اجرا کنید (در همین سشن یا سشن بعدی)، آموزش از آخرین چک‌پوینت ذخیره‌شده ادامه پیدا می‌کند. `SESSION_MINUTES` را متناسب با سقف سشن خودتان تنظیم کنید (Colab رایگان ≈ تا ۶-۱۲ ساعت با قطعی‌های احتمالی، Kaggle رایگان ≈ سقف سشن ۹ ساعت و سهمیه‌ی هفتگی GPU حدود ۳۰ ساعت).

In [ ]:
import re

with open("main_run.py", encoding="utf-8") as f:
    _content = f.read()
_content = re.sub(r"EVAL_EPOCH = \d+", "EVAL_EPOCH = 10", _content)
_content = re.sub(r"MODEL_SAVE_EPOCH = \d+", "MODEL_SAVE_EPOCH = 20", _content)
with open("main_run.py", "w", encoding="utf-8") as f:
    f.write(_content)
print("MODEL_SAVE_EPOCH -> 20 و EVAL_EPOCH -> 10 (ذخیره/ارزیابیِ مکررتر برای سشن‌های محدود)")

In [ ]:
import glob, time

def latest_epoch():
    files = glob.glob("save_weights/contran-*.model")
    if not files:
        return 0
    return max(int(f.split("-")[-1].split(".")[0]) for f in files)

SESSION_MINUTES = 300  # کمی کمتر از سقف واقعی سشن تنظیم کنید تا وقت ذخیره‌ی امن باقی بماند

start_epoch = latest_epoch()
print(f"شروع/ازسرگیری آموزش از epoch {start_epoch}")

proc = subprocess.Popen(
    [sys.executable, "-u", "main_run.py", str(start_epoch)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
t0 = time.time()
try:
    for line in proc.stdout:
        print(line, end="")
        if time.time() - t0 > SESSION_MINUTES * 60:
            print("\n>>> بودجه‌ی زمانی این سشن تمام شد؛ در حال توقف امن آموزش ...")
            proc.terminate()
            break
except KeyboardInterrupt:
    print("متوقف‌شده توسط کاربر.")
    proc.terminate()

proc.wait()
print("آخرین epoch ذخیره‌شده:", latest_epoch())
print("این سلول را در سشن بعدی دوباره اجرا کنید تا آموزش از همین‌جا ادامه پیدا کند.")

## بخش ۵ — ارزیابی

سه معیار روی نویسنده‌های تست (۱۵۰ نویسنده‌ی کاملاً ندیده در آموزش) محاسبه می‌شود:

- **CER تشخیص‌پذیری تولیدشده:** آیا بازشناس داخلی مدل (`model.rec`) می‌تواند تصویر تولیدشده را درست بخواند؟ (پروکسی خوانایی)
- **FID:** فاصله‌ی توزیع تصاویر واقعی و تولیدشده (پروکسی کیفیت بصری کلی)
- **دقت طبقه‌بند سبک نویسنده:** آیا تصویر تولیدشده به‌عنوان همان نویسنده‌ی مقصود طبقه‌بندی می‌شود؟ (پروکسی وفاداری سبک)

In [ ]:
ckpts = sorted(glob.glob("save_weights/contran-*.model"),
                key=lambda f: int(f.split("-")[-1].split(".")[0]))
assert ckpts, "هنوز چک‌پوینتی ذخیره نشده — ابتدا بخش ۴ را اجرا کنید."
ckpt = ckpts[-1]
print("بارگذاری چک‌پوینت:", ckpt)

model = ConTranModel(NUM_WRITERS, show_iter_num=500, oov=True).to(gpu)
model.load_state_dict(torch.load(ckpt, map_location=gpu))
model.eval()

test_loader = torch.utils.data.DataLoader(data_test, batch_size=8, shuffle=False,
                                           collate_fn=sort_batch, num_workers=2)

def generate_images(model, style_imgs, content_labels):
    # style_imgs: b,15,H,W (تصاویر مرجع سبک نویسنده) — content_labels: b,OUTPUT_MAX_LEN
    model.eval()
    with torch.no_grad():
        f_xs = model.gen.enc_image(style_imgs)
        f_xt, f_embed = model.gen.enc_text(content_labels, f_xs.shape)
        f_mix = model.gen.mix(f_xs, f_embed)
        xg = model.gen.decode(f_mix, f_xt)
    return xg

def decode_indices(idx_list):
    from load_data import num_tokens, index2letter, tokens
    idx_list = [i for i in idx_list if i not in (tokens["GO_TOKEN"], tokens["END_TOKEN"], tokens["PAD_TOKEN"])]
    return "".join(index2letter[i - num_tokens] for i in idx_list)

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from load_data import IMG_WIDTH
import Levenshtein as Lev

def to_uint8_rgb(x):
    x = (x.clamp(-1, 1) + 1) / 2
    return x.repeat(1, 3, 1, 1)

fid = FrechetInceptionDistance(feature=192, normalize=True).to(gpu)
n_correct_writer, n_total = 0, 0
cer_edits, cer_lens = 0, 0

with torch.no_grad():
    for batch in test_loader:
        (_, wid, idx, img, img_w, label, img_xt, label_xt, label_xt_swap) = batch
        img, img_xt, label_xt, wid_t = img.to(gpu), img_xt.to(gpu), label_xt.to(gpu), wid.to(gpu)
        xg = generate_images(model, img, label_xt)

        fid.update(to_uint8_rgb(img_xt), real=True)
        fid.update(to_uint8_rgb(xg), real=False)

        pred = model.rec(xg, label_xt, img_width=torch.from_numpy(np.array([IMG_WIDTH] * xg.shape[0])))
        pred_idx = torch.topk(pred, 1, dim=-1)[1].squeeze(-1).cpu().numpy()
        gt_idx = label_xt[:, 1:].cpu().numpy()
        for p, g in zip(pred_idx, gt_idx):
            pt, gt = decode_indices(p.tolist()), decode_indices(g.tolist())
            cer_edits += Lev.distance(pt, gt)
            cer_lens += max(len(gt), 1)

        feat = model.cla.cnn_f(xg)
        logits = model.cla.cnn_c(feat).squeeze(-1).squeeze(-1)
        n_correct_writer += (logits.argmax(dim=-1) == wid_t).sum().item()
        n_total += xg.shape[0]

print(f"FID (واقعی در برابر تولیدشده):            {fid.compute().item():.2f}")
print(f"CER تشخیص‌پذیری تصاویر تولیدشده:          {100*cer_edits/cer_lens:.2f}%")
print(f"دقت طبقه‌بند سبک نویسنده روی تولیدشده‌ها: {100*n_correct_writer/n_total:.2f}%  (شانس تصادفی ≈ {100/NUM_WRITERS:.2f}%)")

## بخش ۶ — دمو: تولید کلمه با سبک یک نویسنده

چند نمونه‌ی «مرجع سبک → واقعی → تولیدشده» را کنار هم نشان می‌دهیم.

In [ ]:
def denorm(t):
    t = (t.clamp(-1, 1) + 1) / 2
    return (t.numpy() * 255).astype("uint8")

batch = next(iter(test_loader))
(_, wid, idx, img, img_w, label, img_xt, label_xt, label_xt_swap) = batch
xg = generate_images(model, img.to(gpu), label_xt.to(gpu)).cpu()

N = 4
fig, axes = plt.subplots(N, 3, figsize=(9, 2.3 * N))
for i in range(N):
    label_text = decode_indices(label_xt[i].tolist())
    axes[i, 0].imshow(denorm(img[i, 0]), cmap="gray")
    axes[i, 0].set_title("نمونه‌ی سبک نویسنده"); axes[i, 0].axis("off")
    axes[i, 1].imshow(denorm(img_xt[i, 0]), cmap="gray")
    axes[i, 1].set_title(f"واقعی: {get_display(ar_reshape(label_text))}"); axes[i, 1].axis("off")
    axes[i, 2].imshow(denorm(xg[i, 0]), cmap="gray")
    axes[i, 2].set_title("تولیدشده"); axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

## بخش ۷ — تجربه‌ی کاربردی: آیا داده‌ی مصنوعی به بازشناسی کمک می‌کند؟

یک **بازشناس مستقل و ساده** (CRNN + CTC، جدا از `model.rec` داخلی GAN) را دو بار آموزش می‌دهیم:
1. فقط با داده‌ی واقعیِ ۳۵۰ نویسنده‌ی train
2. با داده‌ی واقعی + نمونه‌های مصنوعیِ تولیدشده توسط مدل GAN آموزش‌دیده (همان نویسندهای train، محتوای تصادفی از واژگان ۱۲۵ کلمه)

سپس هر دو را روی همان ۱۵۰ نویسنده‌ی test (کاملاً ندیده) ارزیابی و CER/WER را مقایسه می‌کنیم. **این عدد، یافته‌ی اصلی و قابل‌دفاعِ پایان‌نامه است.**

In [ ]:
import torch.nn as nn
from load_data import letter2index, index2letter, num_classes, IMG_HEIGHT

BLANK = num_classes          # اندیس توکن blank برای CTC (بعد از آخرین حرف)
CTC_VOCAB_SIZE = num_classes + 1
MAX_W = 192

class CRNN(nn.Module):
    def __init__(self, n_classes=CTC_VOCAB_SIZE, hidden=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(128, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.rnn = nn.LSTM(128 * 4, hidden, num_layers=2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x):
        feat = self.cnn(x)                                   # b,128,4,W'
        b, c, h, w = feat.shape
        feat = feat.permute(0, 3, 1, 2).reshape(b, w, c * h)  # b,W',128*4
        out, _ = self.rnn(feat)
        return self.fc(out).log_softmax(2)                    # b,W',C

def load_word_image(idx_rel_path, height=IMG_HEIGHT, max_width=MAX_W, base_dir=None):
    path = idx_rel_path if os.path.isabs(idx_rel_path) else os.path.join(base_dir or TARGET_DIR, idx_rel_path + ".png")
    img = cv2.imread(path, 0)
    if img is None:
        return np.zeros((height, max_width), dtype="float32")
    rate = height / img.shape[0]
    img = cv2.resize(img, (int(img.shape[1] * rate) + 1, height), interpolation=cv2.INTER_CUBIC)
    img = img[:, :max_width]
    canvas = np.zeros((height, max_width), dtype="float32")
    canvas[:, :img.shape[1]] = img / 255.0
    return canvas

def encode_label(text, max_chars=12):
    return [letter2index[c] for c in text if c in letter2index][:max_chars]

class WordImageDataset(torch.utils.data.Dataset):
    def __init__(self, manifest):
        self.manifest = manifest  # لیست (idx, label_text)
    def __len__(self):
        return len(self.manifest)
    def __getitem__(self, i):
        idx, label = self.manifest[i]
        img = load_word_image(idx)
        img = (img - 0.5) / 0.5
        return torch.from_numpy(img).unsqueeze(0), torch.tensor(encode_label(label), dtype=torch.long)

class SynthImageDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        self.records = records  # لیست (wid, img_np[H,W] در بازه‌ی 0..1, label_text)
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        _, img, label = self.records[i]
        img = (img.astype("float32") - 0.5) / 0.5
        return torch.from_numpy(img).unsqueeze(0), torch.tensor(encode_label(label), dtype=torch.long)

def ctc_collate(batch):
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs)
    lengths = torch.tensor([len(l) for l in labels])
    labels_cat = torch.cat(labels) if sum(lengths) > 0 else torch.zeros(0, dtype=torch.long)
    return imgs, labels_cat, lengths

def read_manifest(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            idpart, label = line.strip().split(" ", 1)
            _, idx = idpart.split(",")
            out.append((idx, label))
    return out

manifest_train_real = read_manifest("Groundtruth_farsi/gan.aban.tr_va.gt.filter27")
manifest_test_real = read_manifest("Groundtruth_farsi/gan.aban.test.gt.filter27")
print("نمونه‌ی واقعی train/test:", len(manifest_train_real), len(manifest_test_real))

In [ ]:
from load_data import num_tokens

def build_label_xt(text):
    padded = data_train.label_padding(text, num_tokens)
    return torch.tensor(padded, dtype=torch.long)

with open("corpora_farsi/in_vocab.aban.txt", encoding="utf-8") as f:
    aban_vocab_words = [w.strip() for w in f if w.strip()]
print(len(aban_vocab_words), "کلمه در واژگان آبان برای محتوای مصنوعی")

N_SYNTH_PER_WRITER = 2   # چند نمونه‌ی مصنوعی به‌ازای هر نویسنده‌ی train تولید شود (افزایش بدهید اگر GPU/زمان اجازه می‌دهد)

synth_records = []
model.eval()
with torch.no_grad():
    for widx in range(len(data_train)):
        for _ in range(N_SYNTH_PER_WRITER):
            sample = data_train[widx]
            (_, wid, idx, final_img, final_img_width, final_label, img_xt, label_xt, label_xt_swap) = sample
            style_imgs = torch.from_numpy(np.array(final_img)).unsqueeze(0).to(gpu).float()  # 1,15,H,W
            target_word = random.choice(aban_vocab_words)
            content_label = build_label_xt(target_word).unsqueeze(0).to(gpu)
            xg = generate_images(model, style_imgs, content_label)
            img_np = ((xg.squeeze().cpu().numpy().clip(-1, 1) + 1) / 2)  # 0..1
            synth_records.append((wid, img_np, target_word))
        if widx % 50 == 0:
            print(f"{widx}/{len(data_train)} نویسنده پردازش شد")

print("تعداد نمونه‌ی مصنوعی تولیدشده:", len(synth_records))

In [ ]:
def ctc_greedy_decode(logits):
    idx = logits.argmax(-1).cpu().numpy()
    texts = []
    for row in idx:
        chars, prev = [], -1
        for c in row:
            if c != BLANK and c != prev:
                chars.append(index2letter[c])
            prev = c
        texts.append("".join(chars))
    return texts

def train_crnn(dataset, epochs=8, lr=1e-3, batch_size=32, log_every=200):
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True,
                                          collate_fn=ctc_collate, num_workers=2, drop_last=True)
    net = CRNN().to(gpu)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    ctc = nn.CTCLoss(blank=BLANK, zero_infinity=True)
    net.train()
    for ep in range(epochs):
        total = 0.0
        for i, (imgs, labels_cat, lengths) in enumerate(loader):
            imgs = imgs.to(gpu)
            logits = net(imgs)
            logp = logits.permute(1, 0, 2)  # W',b,C — CTC انتظار seq اول دارد
            input_lengths = torch.full((imgs.size(0),), logp.size(0), dtype=torch.long)
            loss = ctc(logp, labels_cat, input_lengths, lengths)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
            if i % log_every == 0:
                print(f"  epoch {ep} iter {i}/{len(loader)} loss={loss.item():.3f}")
        print(f"== epoch {ep}: میانگین loss = {total/len(loader):.3f} ==")
    return net

def evaluate_crnn(net, manifest, batch_size=32):
    ds = WordImageDataset(manifest)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False,
                                          collate_fn=ctc_collate, num_workers=2)
    net.eval()
    ed_sum, len_sum, wer_correct, wer_total, ptr = 0, 0, 0, 0, 0
    with torch.no_grad():
        for imgs, labels_cat, lengths in loader:
            imgs = imgs.to(gpu)
            preds = ctc_greedy_decode(net(imgs))
            gt_texts = []
            for L in lengths.tolist():
                idx_seq = labels_cat[ptr:ptr + L].tolist(); ptr += L
                gt_texts.append("".join(index2letter[c] for c in idx_seq))
            for p, g in zip(preds, gt_texts):
                ed_sum += Lev.distance(p, g)
                len_sum += max(len(g), 1)
                wer_total += 1
                wer_correct += int(p == g)
    return 100 * ed_sum / len_sum, 100 * (1 - wer_correct / wer_total)

In [ ]:
print("=== آموزش با داده‌ی واقعی‌فقط ===")
net_real = train_crnn(WordImageDataset(manifest_train_real), epochs=8)
cer_real, wer_real = evaluate_crnn(net_real, manifest_test_real)
print(f"[واقعی‌فقط]      CER={cer_real:.2f}%  WER={wer_real:.2f}%")

print("\n=== آموزش با داده‌ی واقعی + مصنوعی ===")
combined = torch.utils.data.ConcatDataset([WordImageDataset(manifest_train_real), SynthImageDataset(synth_records)])
net_aug = train_crnn(combined, epochs=8)
cer_aug, wer_aug = evaluate_crnn(net_aug, manifest_test_real)
print(f"[واقعی+مصنوعی]   CER={cer_aug:.2f}%  WER={wer_aug:.2f}%")

print("\n----- نتیجه‌ی نهایی -----")
print(f"CER واقعی‌فقط      : {cer_real:.2f}%")
print(f"CER واقعی+مصنوعی   : {cer_aug:.2f}%")
print(f"تغییر              : {cer_aug - cer_real:+.2f} واحد درصد ({'بهبود' if cer_aug < cer_real else 'بدون بهبود'})")

## بخش ۸ — جمع‌بندی، عیب‌یابی و نکات نگارش پایان‌نامه

### نگهداری چک‌پوینت بین سشن‌ها
- **Colab:** چون `save_weights/logs/imgs` به گوگل‌درایو لینک شده‌اند، به‌صورت خودکار حفظ می‌شوند؛ فقط Drive را در سشن بعد دوباره mount کنید.
- **Kaggle:** پیش از پایان هر سشن، این پوشه‌ها را زیپ و از طریق «Output» نوت‌بوک ذخیره کنید (`!zip -r checkpoint.zip save_weights logs`)، سپس در سشن بعد آن Version را به‌عنوان Dataset اضافه (Add Data) و محتوایش را در `save_weights/` کپی کنید.

### اگر Smoke Test (بخش ۳) خطا داد
- خطای مربوط به `cv2`/`albumentations`: نسخه‌ی opencv را چک کنید؛ `opencv-python-headless` را دوباره نصب کنید.
- خطای CUDA out of memory: `BATCH_SIZE` را در `main_run.py` (خط `BATCH_SIZE = 8`) کم کنید.
- خطای `FileNotFoundError` برای تصاویر: مسیر `TARGET_DIR`/تعداد فایل‌ها را در بخش ۲ دوباره بررسی کنید.
- کد اصلی روی PyTorch نسبتاً قدیمی (۱٫۵) نوشته شده بود؛ روی نسخه‌های جدید معمولاً کار می‌کند چون فقط از APIهای پایه‌ی `nn.Module` استفاده می‌کند، ولی اگر خطای نسخه دیدید همین‌جا در چت بگویید تا اصلاح شود.

### برای نوشتن پایان‌نامه از این نوت‌بوک چه چیزهایی بردارید
1. **جدول اصلی:** CER/WER بازشناسِ «واقعی‌فقط» در برابر «واقعی+مصنوعی» (بخش ۷) — یافته‌ی مرکزی.
2. **جدول ارزیابی مولد:** CER تشخیص‌پذیری، FID، دقت طبقه‌بند سبک (بخش ۵) — کیفیت مدل تولیدی را نشان می‌دهد.
3. **ablation پیشنهادی برای عمق بیشتر:** مقدار `N_SYNTH_PER_WRITER` را تغییر دهید (۱، ۲، ۴، ۸) و ببینید بهبود CER چگونه با حجم داده‌ی مصنوعی تغییر می‌کند — این خودش یک نمودار خوب برای فصل نتایج است.
4. **نمونه‌های کیفی:** گرید بخش ۶ (سبک → واقعی → تولیدشده) برای فصل نتایج و دفاع.
5. **تحلیل خطا:** حروفی که فقط با نقطه فرق دارند (ب/پ/ت/ث، ج/چ/ح/خ) را در خروجی بازشناس بررسی کنید — به احتمال زیاد بیشترین خطاها همین‌جا رخ می‌دهد؛ بحث خوبی برای فصل تحلیل است.